# 💄 Dupe Finder
**Same formula, smaller price.** Find cheaper beauty products whose ingredient list is closest to a product you love.

### How to run it (no coding needed)
1. In the menu, click **Runtime → Run all** (*Exécution → Tout exécuter*).
2. If Google warns that the notebook was not written by Google, click **Run anyway**.
3. Wait 1–3 minutes. At the bottom, a link ending in **.gradio.live** appears: click it to open the app.

If step 2 of the code (the data download) fails, it will ask you to upload `product_info.csv`: click **Choose files** and select it.

*Data: Sephora Products and Skincare Reviews (Kaggle, Nady Inky). Model: sentence-transformers/all-MiniLM-L6-v2 (Hugging Face). Interface: Gradio.*

## Step 1 — Install the tools

In [ ]:
!pip -q install "gradio>=6.0" sentence-transformers kagglehub

## Step 2 — Get the Sephora data from Kaggle

In [ ]:
import os, shutil, glob
if not os.path.exists("product_info.csv"):
    try:
        import kagglehub
        path = kagglehub.dataset_download("nadyinky/sephora-products-and-skincare-reviews")
        shutil.copy(glob.glob(os.path.join(path, "**", "product_info.csv"), recursive=True)[0], "product_info.csv")
        print("Data downloaded from Kaggle.")
    except Exception as e:
        print("Automatic download did not work (", e.__class__.__name__, "). Please upload product_info.csv:")
        from google.colab import files
        files.upload()
print("product_info.csv ready:", os.path.exists("product_info.csv"))

## Step 3 — The matching engine
Cleans ingredient lists, weights ingredients (TF-IDF × position in the list), and compares products with cosine similarity.

In [ ]:
%%writefile dupe_engine.py
"""
Dupe Finder - matching engine
=============================

Finds cheaper beauty products whose formula is close to a given product.

How it works (in plain words):
1. Ingredient lists are cleaned and turned into a list of standard ingredient
   names ("Aqua/Water/Eau" and "Water" become the same thing).
2. Each product becomes a vector. An ingredient weighs more when:
   - it is rare across all products (TF-IDF idea: water or glycerin are in
     everything, so they say little; niacinamide or bakuchiol say a lot);
   - it appears early in the list (EU/US labelling rules list ingredients in
     decreasing order of concentration, so the first ones matter most).
3. Two products are compared with cosine similarity -> a "% formula match".
4. A Hugging Face sentence-embedding model (all-MiniLM-L6-v2) lets users search
   in natural language ("vitamin C serum for dark spots") instead of typing an
   exact product name. If the model cannot be downloaded, a simpler text search
   is used so the app still works offline.

Data: "Sephora Products and Skincare Reviews" (Kaggle, nadyinky), product_info.csv
"""

from __future__ import annotations

import ast
import math
import re
from dataclasses import dataclass

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

# --------------------------------------------------------------------------
# 1. Cleaning ingredient lists
# --------------------------------------------------------------------------

# Ingredients that buyers recognise and that usually drive a product's effect.
# Used only to label "star ingredients" in the results (not in the score).
STAR_INGREDIENTS = {
    "niacinamide": "Niacinamide",
    "retinol": "Retinol",
    "retinal": "Retinal",
    "bakuchiol": "Bakuchiol",
    "ascorbic acid": "Vitamin C",
    "sodium ascorbyl phosphate": "Vitamin C",
    "ascorbyl glucoside": "Vitamin C",
    "tetrahexyldecyl ascorbate": "Vitamin C",
    "3-o-ethyl ascorbic acid": "Vitamin C",
    "magnesium ascorbyl phosphate": "Vitamin C",
    "sodium hyaluronate": "Hyaluronic acid",
    "hyaluronic acid": "Hyaluronic acid",
    "hydrolyzed hyaluronic acid": "Hyaluronic acid",
    "salicylic acid": "Salicylic acid (BHA)",
    "glycolic acid": "Glycolic acid (AHA)",
    "lactic acid": "Lactic acid (AHA)",
    "mandelic acid": "Mandelic acid (AHA)",
    "azelaic acid": "Azelaic acid",
    "ceramide np": "Ceramides",
    "ceramide ap": "Ceramides",
    "ceramide eop": "Ceramides",
    "squalane": "Squalane",
    "centella asiatica extract": "Centella (cica)",
    "madecassoside": "Centella (cica)",
    "panthenol": "Panthenol (B5)",
    "tocopherol": "Vitamin E",
    "zinc oxide": "Zinc oxide (SPF)",
    "titanium dioxide": "Titanium dioxide",
    "caffeine": "Caffeine",
    "peptides": "Peptides",
    "palmitoyl tripeptide-1": "Peptides",
    "palmitoyl tetrapeptide-7": "Peptides",
    "acetyl hexapeptide-8": "Peptides",
    "copper tripeptide-1": "Peptides",
    "allantoin": "Allantoin",
    "urea": "Urea",
    "shea butter": "Shea butter",
    "butyrospermum parkii butter": "Shea butter",
    "snail secretion filtrate": "Snail mucin",
    "tranexamic acid": "Tranexamic acid",
    "alpha-arbutin": "Alpha-arbutin",
    "arbutin": "Arbutin",
    "kojic acid": "Kojic acid",
    "adenosine": "Adenosine",
    "biotin": "Biotin",
    "keratin": "Keratin",
    "hydrolyzed keratin": "Keratin",
    "argania spinosa kernel oil": "Argan oil",
    "simmondsia chinensis seed oil": "Jojoba oil",
    "rosa canina fruit oil": "Rosehip oil",
}

# Common synonyms written in different ways on labels.
SYNONYMS = {
    "aqua": "water", "eau": "water", "aqua/water": "water", "water/aqua": "water",
    "aqua/water/eau": "water", "water/aqua/eau": "water", "purified water": "water",
    "parfum": "fragrance", "fragrance/parfum": "fragrance", "parfum/fragrance": "fragrance",
    "perfume": "fragrance", "aroma": "fragrance",
    "vitamin e": "tocopherol", "vitamin c": "ascorbic acid",
    "vitamin b3": "niacinamide", "vitamin b5": "panthenol",
    "butyrospermum parkii (shea) butter": "shea butter",
    "alcohol denat": "alcohol denat.",
}

_MAY_CONTAIN = re.compile(r"(may contain|\+/-|\[\+/-\]|peut contenir)", re.I)
_PARENS = re.compile(r"\([^)0-9]*\)")   # "(rice)" removed, "(c6-14 olefin)" kept
_SPACES = re.compile(r"\s+")
_TRAILING = re.compile(r"[\s.*†‡®™°]+$")


def _pick_main_list(raw: str) -> str:
    """The CSV stores ingredients as a Python-style list of strings.

    Most products have one string. Kits/palettes have several
    ("Shade A:", "<list>", "Shade B:", "<list>"); we keep the longest list,
    which is usually the main product.
    """
    if not isinstance(raw, str) or not raw.strip():
        return ""
    try:
        parts = ast.literal_eval(raw)
        if isinstance(parts, str):
            parts = [parts]
    except (ValueError, SyntaxError):
        parts = [raw.strip("[]'\"")]
    parts = [p for p in parts if isinstance(p, str)]
    if not parts:
        return ""
    return max(parts, key=lambda p: p.count(","))


def clean_ingredient(token: str) -> str:
    t = token.lower().strip()
    t = t.replace("*", "").replace("®", "").replace("™", "")
    t = _PARENS.sub(" ", t)            # "oryza sativa (rice) bran" -> "oryza sativa bran"
    t = t.replace("–", "-").replace("—", "-")
    t = _SPACES.sub(" ", t).strip()
    t = _TRAILING.sub("", t).strip()
    if "/" in t and len(t) < 40:        # "water/aqua/eau" == "aqua/water/eau"
        parts = [p.strip() for p in t.split("/") if p.strip()]
        if all(p in ("water", "aqua", "eau", "purified water") for p in parts):
            return "water"
        if all(p in ("fragrance", "parfum", "perfume", "aroma") for p in parts):
            return "fragrance"
    t = SYNONYMS.get(t, t)
    return t


def parse_ingredients(raw: str) -> list[str]:
    """Turn a raw ingredient label into a clean, ordered list."""
    text = _pick_main_list(raw)
    if not text:
        return []
    m = _MAY_CONTAIN.search(text)
    if m:
        text = text[: m.start()]
    # split on commas that are not inside parentheses
    tokens, depth, cur = [], 0, []
    for ch in text:
        if ch == "(":
            depth += 1
        elif ch == ")":
            depth = max(0, depth - 1)
        if ch == "," and depth == 0:
            tokens.append("".join(cur))
            cur = []
        else:
            cur.append(ch)
    tokens.append("".join(cur))
    out, seen = [], set()
    for tok in tokens:
        c = clean_ingredient(tok)
        if 2 <= len(c) <= 80 and c not in seen and not c.startswith("please "):
            seen.add(c)
            out.append(c)
    return out


# --------------------------------------------------------------------------
# 2. Size parsing (so we can compare price per ounce, not just price)
# --------------------------------------------------------------------------

_OZ = re.compile(r"(\d+(?:\.\d+)?)\s*(?:fl\.?\s*)?oz", re.I)
_ML = re.compile(r"(\d+(?:\.\d+)?)\s*ml", re.I)
_G = re.compile(r"(\d+(?:\.\d+)?)\s*g\b", re.I)


def size_in_oz(size: str) -> float | None:
    if not isinstance(size, str):
        return None
    for rx, factor in ((_OZ, 1.0), (_ML, 1 / 29.5735), (_G, 1 / 28.3495)):
        m = rx.search(size)
        if m:
            v = float(m.group(1)) * factor
            return v if v > 0 else None
    return None


# --------------------------------------------------------------------------
# 3. The engine
# --------------------------------------------------------------------------

@dataclass
class Match:
    row: pd.Series
    score: float               # 0-100 formula match
    shared_top: list[str]      # shared ingredients, most important first
    shared_stars: list[str]    # shared "star" actives, human-readable
    n_shared: int
    top10_shared: int          # how many of the target's first 10 ingredients it also has

    @property
    def verdict(self) -> str:
        if self.score >= 60:
            return "Near-twin formula"
        if self.score >= 40:
            return "Very close formula"
        if self.score >= 25:
            return "Similar formula"
        return "Loose match"


class DupeFinder:
    EXCLUDED_PRIMARY = {"Mini Size", "Gifts", "Tools & Brushes"}
    EXCLUDED_WORDS = re.compile(r"\b(?:mini|travel size|set|kit|duo|trio|value)\b", re.I)
    STAR_BOOST = 1.5   # active ingredients count a bit more: they drive what a product does

    def __init__(self, csv_path: str = "product_info.csv", use_hf_model: bool = True,
                 hf_model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
        df = pd.read_csv(csv_path)
        df["ingredient_list"] = df["ingredients"].map(parse_ingredients)
        df = df[df["ingredient_list"].map(len) >= 4].copy()
        df = df[~df["primary_category"].isin(self.EXCLUDED_PRIMARY)]
        df = df[~df["product_name"].str.contains(self.EXCLUDED_WORDS, na=False)]
        df["category"] = df["tertiary_category"].fillna(df["secondary_category"]).fillna(
            df["primary_category"])
        df["size_oz"] = df["size"].map(size_in_oz)
        df["price_per_oz"] = np.where(df["size_oz"].notna(), df["price_usd"] / df["size_oz"], np.nan)
        df["label"] = df["brand_name"] + " - " + df["product_name"]
        self.df = df.reset_index(drop=True)
        self._build_formula_matrix()
        self._build_search(use_hf_model, hf_model_name)

    # ---- formula vectors -------------------------------------------------
    def _build_formula_matrix(self):
        lists = self.df["ingredient_list"].tolist()
        vocab: dict[str, int] = {}
        doc_freq: dict[int, int] = {}
        for ings in lists:
            for ing in ings:
                j = vocab.setdefault(ing, len(vocab))
                doc_freq[j] = doc_freq.get(j, 0) + 1
        n = len(lists)
        idf = np.zeros(len(vocab))
        for j, dfreq in doc_freq.items():
            idf[j] = math.log((1 + n) / (1 + dfreq)) + 1.0
        self.vocab, self.idf = vocab, idf
        self.X = sparse.vstack([self._vector_for(ings) for ings in lists]).tocsr()

    def _weight(self, ing: str, rank: int) -> float:
        """Importance of one ingredient in one product."""
        j = self.vocab[ing]
        position_weight = 1.0 / math.sqrt(1 + rank)   # earlier in the list = more concentrated
        boost = self.STAR_BOOST if ing in STAR_INGREDIENTS else 1.0
        return self.idf[j] * position_weight * boost

    def _vector_for(self, ingredients: list[str]):
        cols, vals = [], []
        for rank, ing in enumerate(ingredients):
            if ing in self.vocab:
                cols.append(self.vocab[ing]); vals.append(self._weight(ing, rank))
        v = sparse.csr_matrix((vals, ([0] * len(cols), cols)), shape=(1, len(self.vocab)))
        return normalize(v)

    # ---- product search (Hugging Face model) -----------------------------
    def _search_text(self) -> list[str]:
        hl = self.df["highlights"].fillna("").str.replace(r"[\[\]']", "", regex=True)
        return (self.df["brand_name"] + " " + self.df["product_name"] + ". " +
                self.df["category"].fillna("") + ". " + hl).tolist()

    def _build_search(self, use_hf_model: bool, model_name: str):
        texts = self._search_text()
        self.search_mode = "keyword"
        self.embedder = None
        if use_hf_model:
            try:
                from sentence_transformers import SentenceTransformer
                self.embedder = SentenceTransformer(model_name)
                self.emb = self.embedder.encode(texts, batch_size=128, show_progress_bar=True,
                                                normalize_embeddings=True)
                self.search_mode = "semantic (Hugging Face: " + model_name.split("/")[-1] + ")"
            except Exception as e:  # no internet / library missing -> fallback
                print(f"[DupeFinder] Hugging Face model unavailable ({e.__class__.__name__}); "
                      "using keyword search instead.")
                self.embedder = None
        # keyword search (always built; also used for exact-name matching)
        names = (self.df["brand_name"] + " " + self.df["product_name"]).str.lower().tolist()
        self.tfidf_char = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 4), sublinear_tf=True)
        self.tfidf_word = TfidfVectorizer(analyzer="word", token_pattern=r"[\w\-\.\+]+",
                                          sublinear_tf=True)
        self.Tc = self.tfidf_char.fit_transform(names)
        self.Tw = self.tfidf_word.fit_transform([t.lower() for t in texts])
        self._name_words = [set(re.findall(r"[\w\-\+]+", n)) for n in names]

    def search_products(self, query: str, k: int = 8) -> list[int]:
        """Return row indices of the products that best match a free-text query."""
        query = (query or "").strip()
        if not query:
            return []
        exact = self.df.index[self.df["label"].str.lower() == query.lower()].tolist()
        if exact:
            return exact[:1]
        q = query.lower()
        q_words = set(re.findall(r"[\w\-\+]+", q))
        overlap = np.array([len(q_words & w) / max(1, len(q_words)) for w in self._name_words])
        kw = 0.35 * (self.Tc @ self.tfidf_char.transform([q]).T).toarray().ravel() + \
             0.35 * (self.Tw @ self.tfidf_word.transform([q]).T).toarray().ravel() + \
             0.30 * overlap
        if self.embedder is not None:
            q = self.embedder.encode([query], normalize_embeddings=True)[0]
            sem = self.emb @ q
            scores = 0.6 * sem + 0.4 * kw      # semantic meaning + spelling of names
        else:
            scores = kw
        return list(np.argsort(-scores)[:k])

    # ---- dupes -------------------------------------------------------------
    def _make_match(self, target_ings: list[str], j: int, score: float) -> Match:
        row = self.df.loc[j]
        b = set(row["ingredient_list"])
        shared = [x for x in target_ings if x in b]
        rank = {x: r for r, x in enumerate(target_ings)}
        shared_sorted = sorted(shared, key=lambda x: -self._weight(x, rank[x]))
        stars, seen = [], set()
        for x in shared:
            name = STAR_INGREDIENTS.get(x)
            if name and name not in seen:
                seen.add(name); stars.append(name)
        top10 = sum(1 for x in target_ings[:10] if x in b)
        return Match(row=row, score=score * 100, shared_top=shared_sorted[:6],
                     shared_stars=stars, n_shared=len(shared), top10_shared=top10)

    def _rank(self, target_vec, target_ings, mask, k) -> list[Match]:
        sims = (self.X @ target_vec.T).toarray().ravel()
        mask = mask & (sims < 0.999)   # identical formula = same product re-listed
        cand = np.where(mask)[0]
        out, picked = [], []
        for j in cand[np.argsort(-sims[cand])]:
            # skip near-copies of a result we already show (e.g. same serum, two names)
            if any((self.X[j] @ self.X[p].T).toarray()[0, 0] > 0.97 for p in picked):
                continue
            picked.append(j)
            out.append(self._make_match(target_ings, j, float(sims[j])))
            if len(out) == k:
                break
        return out

    def find_dupes(self, idx: int, k: int = 5, cheaper_only: bool = True,
                   same_category: bool = True, other_brands: bool = True) -> list[Match]:
        target = self.df.loc[idx]
        mask = np.ones(len(self.df), dtype=bool)
        mask[idx] = False
        if same_category:
            mask &= (self.df["category"] == target["category"]).values
        if cheaper_only:
            mask &= (self.df["price_usd"] < target["price_usd"]).values
        if other_brands:
            mask &= (self.df["brand_name"] != target["brand_name"]).values
        return self._rank(self.X[idx], target["ingredient_list"], mask, k)

    def find_from_ingredients(self, raw_text: str, k: int = 5, max_price: float | None = None,
                              category: str | None = None) -> tuple[list[Match], list[str]]:
        """Paste any ingredient list (e.g. from a product not sold at Sephora)."""
        ings = parse_ingredients(str([raw_text]))
        if not ings:
            return [], []
        mask = np.ones(len(self.df), dtype=bool)
        if max_price:
            mask &= (self.df["price_usd"] <= max_price).values
        if category and category != "All categories":
            mask &= (self.df["category"] == category).values
        return self._rank(self._vector_for(ings), ings, mask, k), ings


## Step 4 — The web interface (Gradio)

In [ ]:
%%writefile app.py
"""
Dupe Finder - web app (Gradio)
==============================
Run:  python app.py        (then open the link that appears)
In Google Colab the notebook launches it with a public link.
"""

import html
import os
import urllib.parse

import gradio as gr

from dupe_engine import DupeFinder

CSV_PATH = os.environ.get("DUPE_CSV", "product_info.csv")
SHARE = os.environ.get("DUPE_SHARE", "0") == "1"

print("Loading products and building the formula index...")
finder = DupeFinder(CSV_PATH)
df = finder.df
print(f"Ready: {len(df):,} products, search mode = {finder.search_mode}")

CATEGORIES = ["All categories"] + sorted(df["category"].dropna().unique().tolist())

# ---------------------------------------------------------------------------
# Look & feel
# ---------------------------------------------------------------------------
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Bodoni+Moda:opsz,wght@6..96,500;6..96,700&family=Instrument+Sans:wght@400;500;600&display=swap');
:root{
  --df-bg:#FBF7F9; --df-ink:#2B1B2E; --df-muted:#76657A; --df-line:#E9DDE4;
  --df-lip:#B3124A; --df-blush:#F6D3DF; --df-card:#FFFFFF; --df-good:#1F7A55;
}
.gradio-container{background:var(--df-bg)!important;font-family:'Instrument Sans',system-ui,sans-serif!important;color:var(--df-ink);max-width:1060px!important}
#df-head{padding:28px 4px 8px}
#df-head h1{font-family:'Bodoni Moda',serif;font-weight:700;font-size:clamp(40px,7vw,68px);line-height:1;margin:0;letter-spacing:-.01em;color:var(--df-ink)}
#df-head p{font-size:17px;color:var(--df-muted);max-width:60ch;margin:12px 0 0;line-height:1.5}
.df-target,.df-card,.df-empty,.df-note{color:var(--df-ink)!important}
.df-target *,.df-card *{color:inherit}
.df-target{border-left:4px solid var(--df-ink);padding:10px 16px;margin:6px 0 18px;background:var(--df-card)}
.df-target .t-name{font-family:'Bodoni Moda',serif;font-size:22px;font-weight:500}
.df-target .t-meta{color:var(--df-muted);font-size:14px;margin-top:2px}
.df-card{display:grid;grid-template-columns:118px 1fr;gap:18px;background:var(--df-card);border:1px solid var(--df-line);border-radius:14px;padding:18px 20px;margin-bottom:12px}
.df-score{text-align:center;border-right:1px solid var(--df-line);padding-right:14px}
.df-score .n{font-family:'Bodoni Moda',serif;font-size:44px;line-height:1;color:var(--df-lip)!important;font-weight:700}
.df-score .l{font-size:12px;color:var(--df-muted)!important;margin-top:6px;line-height:1.3}
.df-name{font-size:18px;font-weight:600;line-height:1.3}
.df-name a{color:inherit;text-decoration:none;border-bottom:1px solid var(--df-line)}
.df-name a:hover{border-color:var(--df-lip)}
.df-brand{color:var(--df-muted)!important;font-size:14px}
.df-target .t-meta,.df-inci,.df-note{color:var(--df-muted)!important}
.df-price .save{color:var(--df-good)!important}
.df-inci mark{color:var(--df-ink)!important}
.df-price{display:flex;flex-wrap:wrap;gap:14px;align-items:baseline;margin:8px 0 10px;font-size:15px}
.df-price .p{font-weight:600;font-size:20px}
.df-price .save{color:var(--df-good);font-weight:600}
.df-verdict{font-size:14px;margin-bottom:8px}
.df-chips{display:flex;flex-wrap:wrap;gap:6px;margin:6px 0}
.df-chip{background:var(--df-blush);color:var(--df-ink);border-radius:999px;padding:3px 10px;font-size:13px}
.df-inci{font-size:13px;color:var(--df-muted);line-height:1.55;margin-top:6px}
.df-inci mark{background:transparent;color:var(--df-ink);font-weight:600;border-bottom:2px solid var(--df-blush)}
.df-empty{padding:28px;text-align:center;color:var(--df-muted);border:1px dashed var(--df-line);border-radius:14px}
.df-note{font-size:13px;color:var(--df-muted);margin-top:14px;line-height:1.5}
button.primary{background:var(--df-lip)!important;border-color:var(--df-lip)!important}
@media (max-width:640px){.df-card{grid-template-columns:1fr}.df-score{border-right:0;border-bottom:1px solid var(--df-line);padding:0 0 10px;text-align:left;display:flex;gap:12px;align-items:baseline}}
"""

THEME = gr.themes.Soft(primary_hue="pink", neutral_hue="stone",
                       font=[gr.themes.GoogleFont("Instrument Sans"), "system-ui", "sans-serif"])


# ---------------------------------------------------------------------------
# HTML helpers
# ---------------------------------------------------------------------------
def esc(x) -> str:
    return html.escape(str(x))


def money(x) -> str:
    return f"${x:,.2f}".replace(".00", "")


def sephora_link(row) -> str:
    q = urllib.parse.quote_plus(f"{row['brand_name']} {row['product_name']}")
    return f"https://www.sephora.com/search?keyword={q}"


def target_html(row) -> str:
    size = f" · {esc(row['size'])}" if isinstance(row["size"], str) else ""
    return (f"<div class='df-target'><div class='t-name'>{esc(row['product_name'])}</div>"
            f"<div class='t-meta'>{esc(row['brand_name'])} · {money(row['price_usd'])}{size} · "
            f"{esc(row['category'])} · {len(row['ingredient_list'])} ingredients</div></div>")


def card_html(m, target_price=None, target_ppo=None) -> str:
    r = m.row
    price_line = f"<span class='p'>{money(r['price_usd'])}</span>"
    if isinstance(r["size"], str):
        price_line += f"<span>{esc(r['size'])}</span>"
    if target_price:
        diff = target_price - r["price_usd"]
        if diff > 0:
            price_line += f"<span class='save'>{money(diff)} cheaper ({diff / target_price:.0%} less)</span>"
    if target_ppo and r["price_per_oz"] == r["price_per_oz"]:   # not NaN
        price_line += f"<span>{money(r['price_per_oz'])}/oz vs {money(target_ppo)}/oz</span>"
    chips = "".join(f"<span class='df-chip'>{esc(s)}</span>" for s in m.shared_stars)
    chips = f"<div class='df-chips'>{chips}</div>" if chips else ""
    shared = ", ".join(f"<mark>{esc(x)}</mark>" for x in m.shared_top)
    return f"""
<div class='df-card'>
  <div class='df-score'><div class='n'>{m.score:.0f}%</div><div class='l'>formula match</div></div>
  <div>
    <div class='df-name'><a href='{sephora_link(r)}' target='_blank' rel='noopener'>{esc(r['product_name'])}</a></div>
    <div class='df-brand'>{esc(r['brand_name'])}</div>
    <div class='df-price'>{price_line}</div>
    <div class='df-verdict'><b>{m.verdict}.</b> Shares {m.n_shared} ingredients, including
      {m.top10_shared} of the original's first 10 (the most concentrated ones).</div>
    {chips}
    <div class='df-inci'>Most important shared ingredients: {shared or 'none'}</div>
  </div>
</div>"""


NOTE = ("<div class='df-note'>The match score compares ingredient lists, weighting rare and "
        "highly concentrated ingredients more. It does not know exact percentages, textures or "
        "fragrances, so read it as a shortlist to test, not a guarantee. Prices are from the "
        "Sephora US catalogue (2023).</div>")


def empty(msg: str) -> str:
    return f"<div class='df-empty'>{esc(msg)}</div>"


# ---------------------------------------------------------------------------
# Callbacks
# ---------------------------------------------------------------------------
def option_label(i: int) -> str:
    r = df.loc[i]
    return f"{r['brand_name']} - {r['product_name']} ({money(r['price_usd'])})"


def search(query):
    ids = finder.search_products(query, k=10)
    if not ids:
        return gr.update(choices=[], value=None), empty("Type a product, a brand or what it does.")
    choices = [(option_label(i), int(i)) for i in ids]
    return (gr.update(choices=choices, value=int(ids[0])),
            empty("Check the product picked below, then select Find dupes."))


def find(idx, cheaper, same_cat, other_brands, k):
    if idx is None:
        return empty("Search for a product first, then pick it from the list.")
    idx = int(idx)
    target = df.loc[idx]
    matches = finder.find_dupes(idx, k=int(k), cheaper_only=cheaper,
                                same_category=same_cat, other_brands=other_brands)
    head = target_html(target)
    if not matches:
        return head + empty("No cheaper match in this category. Try unticking a filter.")
    ppo = target["price_per_oz"] if target["price_per_oz"] == target["price_per_oz"] else None
    cards = "".join(card_html(m, target["price_usd"], ppo) for m in matches)
    return head + cards + NOTE


def from_ingredients(text, max_price, category, k):
    if not text or text.count(",") < 3:
        return empty("Paste a full ingredient list, separated by commas (at least 4 ingredients).")
    matches, ings = finder.find_from_ingredients(text, k=int(k),
                                                 max_price=max_price or None, category=category)
    known = sum(1 for x in ings if x in finder.vocab)
    head = (f"<div class='df-target'><div class='t-name'>Your ingredient list</div>"
            f"<div class='t-meta'>{len(ings)} ingredients read, {known} found in the catalogue</div></div>")
    if not matches:
        return head + empty("Nothing matches these filters. Raise the price limit or pick All categories.")
    return head + "".join(card_html(m) for m in matches) + NOTE


# ---------------------------------------------------------------------------
# Layout
# ---------------------------------------------------------------------------
EXAMPLE_INCI = ("Water, Glycerin, Niacinamide, Butylene Glycol, Pentylene Glycol, Sodium Hyaluronate, "
                "Panthenol, Allantoin, Ceramide NP, Carbomer, Sodium Hydroxide, Phenoxyethanol, "
                "Ethylhexylglycerin")

with gr.Blocks(title="Dupe Finder") as demo:
    gr.HTML("<div id='df-head'><h1>Dupe Finder</h1>"
            "<p>Same formula, smaller price. Pick a beauty product you love and see which cheaper "
            f"products share its ingredients. {len(df):,} Sephora products compared by AI.</p></div>")

    with gr.Tab("Find a dupe"):
        with gr.Row():
            query = gr.Textbox(label="Product you love",
                               placeholder="e.g. Tatcha Water Cream, or 'vitamin C serum for dark spots'",
                               scale=4)
            search_btn = gr.Button("Search", scale=1)
        pick = gr.Dropdown(label="Choose the exact product", choices=[], interactive=True)
        with gr.Accordion("Filters", open=False):
            with gr.Row():
                cheaper = gr.Checkbox(value=True, label="Only cheaper products")
                same_cat = gr.Checkbox(value=True, label="Same type of product")
                other_brands = gr.Checkbox(value=True, label="Other brands only")
                k = gr.Slider(3, 10, value=5, step=1, label="Number of dupes")
        go = gr.Button("Find dupes", variant="primary")
        out = gr.HTML(empty("Start by typing a product you love."))
        gr.Examples([["Tatcha The Water Cream"], ["Drunk Elephant Protini moisturizer"],
                     ["Charlotte Tilbury Magic Cream Moisturizer"], ["La Mer Moisturizing Soft Cream"],
                     ["Glow Recipe Watermelon Glow AHA"]], inputs=[query], label="Try one",
                    fn=search, outputs=[pick, out], run_on_click=True, cache_examples=False)

        search_btn.click(search, query, [pick, out])
        query.submit(search, query, [pick, out])
        go.click(find, [pick, cheaper, same_cat, other_brands, k], out)

    with gr.Tab("Paste an ingredient list"):
        gr.Markdown("Found a product somewhere else (a pharmacy brand, a gift, a TikTok find)? "
                    "Paste its ingredient list from the box or the brand's website to see which "
                    "Sephora products have the closest formula.")
        inci = gr.Textbox(label="Ingredient list", lines=5, value=EXAMPLE_INCI)
        with gr.Row():
            max_price = gr.Number(label="Maximum price ($, optional)", value=None)
            cat = gr.Dropdown(CATEGORIES, value="All categories", label="Type of product")
            k2 = gr.Slider(3, 10, value=5, step=1, label="Number of results")
        go2 = gr.Button("Find closest formulas", variant="primary")
        out2 = gr.HTML()
        go2.click(from_ingredients, [inci, max_price, cat, k2], out2)

    with gr.Tab("How it works"):
        gr.Markdown(f"""
**1. Read the label.** Every product's ingredient list is cleaned so that the same ingredient
written differently ("Aqua/Water/Eau" and "Water") counts once.

**2. Weigh what matters.** Ingredients are listed from most to least concentrated, so the first
ones weigh more. Rare ingredients (niacinamide, peptides, retinol) weigh more than ingredients
found in almost everything (water, glycerin): this is the TF-IDF idea from text mining. Known
active ingredients get a small extra boost.

**3. Compare formulas.** Each product becomes a vector of weighted ingredients, and products are
compared with cosine similarity, giving the % formula match.

**4. Understand the request.** The search box uses a Hugging Face sentence-embedding model
(all-MiniLM-L6-v2), so you can describe a product in your own words. Current search mode:
*{finder.search_mode}*.

Data: *Sephora Products and Skincare Reviews* (Kaggle, Nady Inky, 2023), {len(df):,} products
with complete ingredient lists after cleaning.
""")

# Always open in light mode, even when the computer is in dark mode
FORCE_LIGHT = """() => {
  const url = new URL(window.location);
  if (url.searchParams.get('__theme') !== 'light') {
    url.searchParams.set('__theme', 'light');
    window.location.href = url.href;
  }
}"""

if __name__ == "__main__":
    demo.launch(share=SHARE, css=CSS, theme=THEME, js=FORCE_LIGHT)


## Step 5 — Launch the app
Click the **https://…gradio.live** link that appears below. The link works for 72 hours; to stop the app, click the ■ stop button of this cell.

In [ ]:
import os
os.environ["DUPE_SHARE"] = "1"
!python app.py